# Análise Comparativa do Salário Médio Mensal (Trabalhadores Formais): Paraíba vs. Rio Grande do Norte

Este notebook realiza um estudo de inferência estatística para verificar se existe diferença significativa entre as médias do **Salário médio mensal dos trabalhadores formais** (em salários mínimos) dos municípios da **Paraíba (PB)** e do **Rio Grande do Norte (RN)**, considerando um nível de significância de $\alpha = 5\%$ ($0,05$).

## Etapas da Análise
1. **Carregamento e Inspeção:** Leitura das bases municipais completas ($N_{PB} = 223$, $N_{RN} = 167$).
2. **Análise Exploratória e Visualização:** Cálculo das medidas de tendência central e dispersão, acompanhado de Boxplot e Histograma/KDE.
3. **Teste de Homogeneidade de Variâncias (Suposições):** Avaliação da igualdade das variâncias via **Teste F de Snedecor** e **Teste de Levene** ($H_0: \sigma_{PB}^2 = \sigma_{RN}^2$ vs. $H_1: \sigma_{PB}^2 \neq \sigma_{RN}^2$).
4. **Teste de Hipóteses para Duas Médias:** Aplicação do **Teste $t$ de Welch** (devido à desigualdade das variâncias) para testar $H_0: \mu_{PB} = \mu_{RN}$ contra $H_1: \mu_{PB} \neq \mu_{RN}$.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

## 1. Carregamento e Preparação dos Dados

In [2]:
df_pb = pd.read_excel("Dados Paraíba.xlsx")
df_rn = pd.read_excel("Dados RN.xlsx")

col_sal = "Salário médio mensal dos trabalhadores formais"

sal_pb = pd.to_numeric(df_pb[col_sal], errors="coerce").dropna()
sal_rn = pd.to_numeric(df_rn[col_sal], errors="coerce").dropna()

print(f"Municípios válidos na Paraíba (PB): {len(sal_pb)}")
print(f"Municípios válidos no Rio Grande do Norte (RN): {len(sal_rn)}")

Municípios válidos na Paraíba (PB): 223
Municípios válidos no Rio Grande do Norte (RN): 167

## 2. Estatísticas Descritivas e Visualização

In [3]:
resumo = pd.DataFrame({
    "Estado": ["Paraíba (PB)", "Rio Grande do Norte (RN)"],
    "N (Municípios)": [len(sal_pb), len(sal_rn)],
    "Média (SM)": [sal_pb.mean(), sal_rn.mean()],
    "Mediana (SM)": [sal_pb.median(), sal_rn.median()],
    "Desvio Padrão": [sal_pb.std(ddof=1), sal_rn.std(ddof=1)],
    "Variância": [sal_pb.var(ddof=1), sal_rn.var(ddof=1)],
    "Mínimo": [sal_pb.min(), sal_rn.min()],
    "Máximo": [sal_pb.max(), sal_rn.max()]
})

print(resumo.round(4).to_string(index=False))

                  Estado  N (Municípios)  Média (SM)  Mediana (SM)  Desvio Padrão  Variância  Mínimo  Máximo
            Paraíba (PB)             223      1.8009           1.8         0.2270     0.0515     1.2     2.7
Rio Grande do Norte (RN)             167      1.8317           1.8         0.3079     0.0948     1.2     3.3

In [4]:
df_plot = pd.DataFrame({
    "Salario_Medio": pd.concat([sal_pb, sal_rn], ignore_index=True),
    "Estado": ["Paraíba"] * len(sal_pb) + ["Rio Grande do Norte"] * len(sal_rn)
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_plot, x="Estado", y="Salario_Medio", ax=axes[0], width=0.45, palette=["#1b9e77", "#7570b3"])
axes[0].set_title("Distribuição do Salário Médio por Estado (Boxplot)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Salário Médio Mensal (Salários Mínimos)")
axes[0].set_xlabel("")

sns.histplot(data=df_plot, x="Salario_Medio", hue="Estado", kde=True, element="step", ax=axes[1], palette=["#1b9e77", "#7570b3"])
axes[1].axvline(sal_pb.mean(), color="#1b9e77", linestyle="--", label=f"Média PB ({sal_pb.mean():.2f} SM)")
axes[1].axvline(sal_rn.mean(), color="#7570b3", linestyle="--", label=f"Média RN ({sal_rn.mean():.2f} SM)")
axes[1].set_title("Histograma e Densidade do Salário Médio", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Salário Médio Mensal (Salários Mínimos)")
axes[1].set_ylabel("Frequência")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Verificação das Suposições sobre as Variâncias

Antes de escolher o teste de comparação de médias, avaliamos a homogeneidade das variâncias entre os dois estados ao nível $\alpha = 0,05$:

* **Hipótese Nula ($H_0$):** $\sigma_{PB}^2 = \sigma_{RN}^2$ (as variâncias dos salários médios municipais são iguais).
* **Hipótese Alternativa ($H_1$):** $\sigma_{PB}^2 \neq \sigma_{RN}^2$ (as variâncias dos salários médios municipais são diferentes).

In [5]:
alpha = 0.05

# 1. Teste F de Snedecor (bicaudal)
var_pb, var_rn = sal_pb.var(ddof=1), sal_rn.var(ddof=1)
n_pb, n_rn = len(sal_pb), len(sal_rn)

F_stat = var_pb / var_rn
gl_num, gl_den = n_pb - 1, n_rn - 1
p_valor_F = 2 * min(stats.f.cdf(F_stat, gl_num, gl_den), 1 - stats.f.cdf(F_stat, gl_num, gl_den))

# 2. Teste de Levene (robusto à assimetria/outliers)
W_stat, p_valor_levene = stats.levene(sal_pb, sal_rn, center="median")

print(f"Teste F: Estatística F = {F_stat:.4f} | gl = ({gl_num}, {gl_den}) | p-valor = {p_valor_F:.6f}")
print(f"Teste de Levene: Estatística W = {W_stat:.4f} | p-valor = {p_valor_levene:.4f}")

if p_valor_F < alpha and p_valor_levene < alpha:
    print("\nConclusão sobre variâncias: Rejeita-se H0 (p < 0,05). As variâncias são significativamente DIFERENTES.")
    print("Procedimento indicado: Teste t de Welch (equal_var=False).")
else:
    print("\nConclusão sobre variâncias: Não se rejeita H0. Assume-se variâncias IGUAIS (Teste t de Student).")

Teste F: Estatística F = 0.5434 | gl = (222, 166) | p-valor = 0.000023
Teste de Levene: Estatística W = 8.6803 | p-valor = 0.0034

Conclusão sobre variâncias: Rejeita-se H0 (p < 0,05). As variâncias são significativamente DIFERENTES.
Procedimento indicado: Teste t de Welch (equal_var=False).

## 4. Teste de Hipóteses para Comparação das Médias

Como os testes de variância indicam **heterocedasticidade** ($s_{PB}^2 = 0,0515$ vs. $s_{RN}^2 = 0,0948$; $p < 0,0001$ no Teste F e $p = 0,0034$ no Teste de Levene), utilizamos o **Teste $t$ de Welch** para amostras independentes com variâncias desiguais:

* **Hipótese Nula ($H_0$):** $\mu_{PB} = \mu_{RN}$ (não há diferença significativa entre as médias salariais dos municípios da PB e do RN).
* **Hipótese Alternativa ($H_1$):** $\mu_{PB} \neq \mu_{RN}$ (há diferença significativa entre as médias salariais dos municípios da PB e do RN).

In [6]:
t_stat, p_valor_t = stats.ttest_ind(sal_pb, sal_rn, equal_var=False)

# Graus de liberdade aproximados de Welch-Satterthwaite
s1_n1 = var_pb / n_pb
s2_n2 = var_rn / n_rn
gl_welch = (s1_n1 + s2_n2)**2 / ((s1_n1**2) / (n_pb - 1) + (s2_n2**2) / (n_rn - 1))

# Intervalo de Confiança de 95% para a diferença das médias (PB - RN)
diff_medias = sal_pb.mean() - sal_rn.mean()
erro_padrao_diff = np.sqrt(s1_n1 + s2_n2)
ic_95 = stats.t.interval(0.95, df=gl_welch, loc=diff_medias, scale=erro_padrao_diff)

print(f"Estatística t (Welch): {t_stat:.4f}")
print(f"Graus de liberdade (Welch): {gl_welch:.2f}")
print(f"p-valor (bicaudal): {p_valor_t:.4f}")
print(f"Diferença entre as médias (PB - RN): {diff_medias:.4f} SM")
print(f"IC 95% para a diferença: [{ic_95[0]:.4f}, {ic_95[1]:.4f}] SM")

Estatística t (Welch): -1.0911
Graus de liberdade (Welch): 292.39
p-valor (bicaudal): 0.2761
Diferença entre as médias (PB - RN): -0.0308 SM
IC 95% para a diferença: [-0.0865, 0.0248] SM

## 5. Interpretação dos Resultados

1. **Suposição das Variâncias:** Tanto o Teste F ($F = 0,5434; p = 0,000023$) quanto o Teste de Levene ($W = 8,6803; p = 0,0034$) rejeitam a hipótese nula de igualdade de variâncias ao nível de 5%. O Rio Grande do Norte possui variância salarial quase duas vezes maior ($s^2 = 0,0948$) que a da Paraíba ($s^2 = 0,0515$), refletindo maior dispersão e presença de municípios com médias salariais mais elevadas (até $3,3$ SM, contra máximo de $2,7$ SM na PB). Portanto, o **Teste $t$ de Welch** é o procedimento adequado.
2. **Comparação das Médias:** O Teste $t$ de Welch apresentou $t = -1,0911$ com $p\text{-valor} = 0,2761$ ($p > 0,05$). Como o valor-p é superior ao nível de significância de 5% (e o IC 95% de $[-0,0865; 0,0248]$ inclui o zero), **não rejeitamos a hipótese nula ($H_0$)**.
3. **Conclusão Prática:** Não há evidência estatística de diferença significativa entre a média do salário médio mensal dos trabalhadores formais dos municípios da Paraíba ($1,80$ salários mínimos) e do Rio Grande do Norte ($1,83$ salários mínimos).